# ColdSite-DTI — configurable binary grid (Kaggle)

One notebook for any of the four baseline/ours models, on either dataset, over any subset
of the four split types -- so two Kaggle accounts (or two commits of the same account) can
each own a share of the work without ever training the same cell twice.

| model | role |
|---|---|
| DeepDTA | accuracy anchor (no attention, never audited) |
| ColdSite-DTI | our model |
| HyperAttentionDTI | published model under audit |
| MolTrans | published model under audit (deferred until this notebook existed) |

**Why binary.** DeepDTA, HyperAttentionDTI and MolTrans only predict binding / non-binding
in this project's recipe, so the audit table can only compare models on one metric --
AUROC -- if every model does the same task. A pair counts as binding at
**pKd >= threshold** (`BINARY_THRESHOLD` in `src/model/dataset.py`: DAVIS 7.0, KIBA 12.1).

## Before you run

| Setting | Value |
|---|---|
| Accelerator | **GPU** (T4 x2 if available -- more GPUs, more parallel splits) |
| Internet | **On** |
| Environment | **Pin to original** |
| How to run | **Save Version -> Save & Run All (Commit)** |
| Settings cell | edit `DATASET`, `MODELS`, `SPLIT_SUBSET`, `SEEDS` below -- see presets there |

## The loop you will repeat -- read this

Every commit starts in a **fresh, empty container**. Finished cells do not carry over by
themselves. So:

1. **First commit:** leave `RESTORE_FROM = None` and commit.
2. When it finishes: open that version -> **Output** -> download the results zip, and
   upload it as a Kaggle **Dataset** (name it so you know which account/model/dataset it
   is -- e.g. `coldsite-moltrans-davis-results`).
3. **Every later commit:** attach that dataset (Add Input), set `RESTORE_FROM` in the
   restore cell, and commit. Finished cells are skipped; interrupted ones are retrained.
4. After each commit, add the new output to the dataset (**New Version** of it), so the
   next restore has everything.

**The notebook stops itself at 11 hours.** Kaggle ends a commit at 12, and a commit cut off
mid-cell may not save its output. A cell stopped mid-training is retrained on the next
commit.

**MolTrans checkpoints are big** -- about 250 MB each (vs. ~2.5 MB for the other three
models; its decoder has a 78,192 -> 512 linear layer). A 12-cell MolTrans grid is ~3 GB.
Plan the restore-dataset size and the Drive backup accordingly.


## 1. Settings — the only cell you should need to edit

In [ ]:
# ============================================================================
# SETTINGS
# ============================================================================

DATASET = 'davis'                 # 'davis' | 'kiba'
MODELS = ['moltrans']             # any subset of: 'deepdta', 'coldsite_dti', 'hyperattentiondti', 'moltrans'
SPLIT_SUBSET = ['random', 'cold_drug', 'cold_target', 'cold_pair']  # which split types THIS run trains
SEEDS = [1, 2, 3]

# --- presets, for reference -- delete or ignore whichever you are not using ------------
# Account 2, MolTrans on DAVIS (first job -- all four splits, nobody else is training them):
#   DATASET = 'davis'; MODELS = ['moltrans']
#   SPLIT_SUBSET = ['random', 'cold_drug', 'cold_target', 'cold_pair']
#
# Account 2, MolTrans on KIBA (after DAVIS is done -- HALF the splits; agree the other
# half with account 1 first, see CLAUDE.md):
#   DATASET = 'kiba'; MODELS = ['moltrans']
#   SPLIT_SUBSET = ['random', 'cold_drug']                      # example half
#
# Account 1, once the DAVIS 36-grid is 36/36 -- KIBA, the three DAVIS-grid models, the
# OTHER half of the splits:
#   DATASET = 'kiba'; MODELS = ['deepdta', 'coldsite_dti', 'hyperattentiondti']
#   SPLIT_SUBSET = ['cold_target', 'cold_pair']                 # the half account 2 is not training
# -----------------------------------------------------------------------------------------

ALL_SPLITS = ['random', 'cold_drug', 'cold_target', 'cold_pair']
KNOWN_MODELS = ['deepdta', 'coldsite_dti', 'hyperattentiondti', 'moltrans']

assert DATASET in ('davis', 'kiba'), DATASET
assert MODELS and all(m in KNOWN_MODELS for m in MODELS), \
    f'MODELS must be a non-empty subset of {KNOWN_MODELS}, got {MODELS}'
assert SPLIT_SUBSET and all(s in ALL_SPLITS for s in SPLIT_SUBSET), \
    f'SPLIT_SUBSET must be a non-empty subset of {ALL_SPLITS}, got {SPLIT_SUBSET}'
assert SEEDS, 'SEEDS must not be empty'

TOTAL_CELLS = len(MODELS) * len(SPLIT_SUBSET) * len(SEEDS)
print(f'{DATASET} | models: {", ".join(MODELS)} | splits: {", ".join(SPLIT_SUBSET)} | seeds: {SEEDS}')
print(f'{TOTAL_CELLS} cells this run ({len(MODELS)} model(s) x {len(SPLIT_SUBSET)} split(s) x {len(SEEDS)} seed(s))')


## 2. Check the GPU(s)

In [ ]:
import time
START = time.time()          # the 11-hour self-stop is measured from here

import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU.'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)

# Measured peak memory on a 1000-residue protein (STATUS.md), from the DAVIS 36-grid:
#   ColdSite-DTI  batch 64 -> 8.7 GB, 16 -> 2.3 GB
#   HyperAttentionDTI  batch 32 -> ~5 GB; 8 x accum 4 is the same effective batch
#   DeepDTA  batch 256 -> 0.7 GB
#   MolTrans  not yet profiled on a full split -- stays at the vendored batch size (16)
#   rather than guessing a larger one on a bigger GPU.
big = torch.cuda.get_device_properties(0).total_memory / 1e9 >= 14
COLDSITE_BATCH = 64 if big else 16
HAT_BATCH, HAT_ACCUM = (32, 1) if big else (8, 4)
DEEPDTA_BATCH = 256
MOLTRANS_BATCH = 16
print(f'batches: ColdSite {COLDSITE_BATCH}, HAT {HAT_BATCH}x{HAT_ACCUM}, '
      f'DeepDTA {DEEPDTA_BATCH}, MolTrans {MOLTRANS_BATCH}')


## 3. Clone the repo

Pinned to `main` on the fork. If this run trains MolTrans, the cell also refuses to
continue on a checkout that predates `src/model/train_moltrans.py` -- that file has to be
pushed to `origin/main` before Kaggle can clone it.

In [ ]:
import os

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC  = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q tabulate subword-nmt

import importlib, src.model.dataset as _ds
importlib.reload(_ds)
assert hasattr(_ds, 'BINARY_THRESHOLD'), (
    'This checkout predates the binary-label fix -- ColdSite-DTI would crash. '
    'Re-run this cell so git pull fetches main.')
if 'moltrans' in MODELS:
    assert os.path.exists('src/model/train_moltrans.py'), (
        'This checkout has no src/model/train_moltrans.py -- push that commit to '
        'origin/main before running MolTrans here, then re-run this cell.')

RESULTS = f'{WORK}/results'
os.makedirs(RESULTS, exist_ok=True)
print()
!git log --oneline -1
print('results ->', RESULTS)


## 4. Fetch the DeepDTA source files

`build_splits` needs these for both datasets regardless of which models this run trains.

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
!python -m src.data.load_data


## 5. Build the splits — and verify they match

Every cell in this project has to come from the same splits, whichever account or model
trains it. The counts below were verified on a MacBook, on Colab and on Kaggle.

In [ ]:
!python -m src.data.build_splits 2>&1 | grep -E 'davis|kiba|leakage'

import pandas as pd

EXPECTED = {
    'davis': {
        'random':      (21039, 3006, 6011),
        'cold_drug':   (21658, 2652, 5746),
        'cold_target': (21080, 2992, 5984),
        'cold_pair':   (15190,  264, 1144),
    },
    'kiba': {
        'random':      (82778, 11825, 23651),
        'cold_drug':   (83807, 12073, 22374),
        'cold_target': (85452, 10701, 22101),
        'cold_pair':   (58041,  1334,  4375),
    },
}

problems = []
for split in ALL_SPLITS:
    expected = EXPECTED[DATASET][split]
    got = tuple(len(pd.read_csv(f'data/splits/{DATASET}/{split}/{part}.csv'))
                for part in ('train', 'valid', 'test'))
    marker = 'OK' if got == expected else 'MISMATCH'
    flag = '  <- this run trains it' if split in SPLIT_SUBSET else ''
    if got != expected:
        problems.append(f'{split}: expected {expected}, got {got}')
    print(f"{split:12s} {str(got):26s} {marker}{flag}")
assert not problems, f'{DATASET} splits differ from the rest of the project:\n  ' + '\n  '.join(problems)
print(f'\n{DATASET} splits match. Safe to train.')


## 6. Restore results from a previous commit

`None` on the first commit. After that, attach the dataset holding the previous output and
put its path here. Only model files are copied, and nothing already present is
overwritten.

In [ ]:
import shutil, glob

# '/kaggle/input' -- searches every attached input. Kaggle does not always mount a
# dataset at /kaggle/input/<name> (2026-09-12: that path failed; this one restored 65).
RESTORE_FROM = None

if RESTORE_FROM:
    assert os.path.isdir(RESTORE_FROM), f'not a directory: {RESTORE_FROM}'
    copied = skipped = 0
    for src in glob.glob(f'{RESTORE_FROM}/**/*', recursive=True):
        name = os.path.basename(src)
        if not name.endswith(('.pt', '_results.json', '_history.json')):
            continue
        dst = os.path.join(RESULTS, name)
        if os.path.exists(dst):
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1
    print(f'restored {copied} file(s), left {skipped} already present')
else:
    print('RESTORE_FROM is None -- starting from an empty results folder.')


## 7. The runner

Each GPU works through its own queue of commands in order; the GPUs run in parallel. This
run's `SPLIT_SUBSET` is divided across the available GPUs; each GPU trains every model in
`MODELS`, cheapest first, over its share of splits. Every line is prefixed with where it
comes from -- GPU, model, split, seed and cell number -- so a line never needs scrolling
back to place. Each finished cell gets a `✓` line with its test AUROC and the running total,
and every 10 minutes a `STATUS` line shows every GPU at once: search the log for `STATUS`.

At the 11-hour mark it stops launching new work and ends anything still running, so the
commit can save.

In [ ]:
import subprocess, threading, glob, json, re
from src.model.checkpoint_naming import checkpoint_path, results_path, run_tag

DEADLINE = START + 11 * 3600
STATUS_EVERY = 600          # seconds between STATUS lines

splits_list = list(SPLIT_SUBSET)
SHARE = {str(g): splits_list[g::N_GPU] for g in range(N_GPU)}
SHARE = {g: s for g, s in SHARE.items() if s}   # drop a GPU with nothing to do

MODEL_NAME = {'src.model.train_deepdta': 'DeepDTA', 'src.model.run_grid': 'ColdSite',
              'src.model.train_hyperattentiondti': 'HAT', 'src.model.train_moltrans': 'MolTrans'}
HEADER = re.compile(r'^\w+/(\w+)/seed(\d+)\s*$')          # run_grid: a cell starts
SKIPPED = re.compile(r'^\[skip\] \w+/(\w+)/seed(\d+)')     # run_grid: a cell was done
EPOCH = re.compile(r'^\s*(?:epoch|Epoch)\s+(\d+)')
SAVED = re.compile(r'Saved -> (\S+_results\.json)')


def hours_left():
    return (DEADLINE - time.time()) / 3600


def cells_done():
    done = 0
    for model in MODELS:
        for split in SPLIT_SUBSET:
            for seed in SEEDS:
                tag = run_tag(DATASET, split, 'binary', seed)
                if os.path.exists(results_path(RESULTS, tag, model=model)):
                    done += 1
    return done


def _arg(cmd, flag):
    return cmd[cmd.index(flag) + 1] if flag in cmd else '-'


def _cells_in(cmd):
    if cmd[3] == 'src.model.run_grid':
        return len(_arg(cmd, '--splits').split(',')) * len(_arg(cmd, '--seeds').split(','))
    return 1


def run_parallel(queues, label):
    """queues: {gpu: [command, ...]}. Returns True if the deadline cut it short."""
    current, state = {}, {'deadline': False}
    where = {g: {'model': '-', 'split': '-', 'seed': '-', 'epoch': '-', 'cell': 0,
                 'total': sum(_cells_in(c) for c in q)} for g, q in queues.items()}

    def tag(g):
        w = where[g]
        return f"[GPU{g} {w['model']} {w['split']} s{w['seed']} · cell {w['cell']}/{w['total']}]"

    def begin_cell(g, split, seed):
        where[g].update(split=split, seed=seed, epoch='-')
        where[g]['cell'] += 1

    def worker(gpu, commands):
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': gpu, 'PYTHONUNBUFFERED': '1'}
        with open(f'{WORK}/{label}_gpu{gpu}.log', 'a') as log:
            for cmd in commands:
                if time.time() > DEADLINE:
                    return
                module = cmd[3]
                where[gpu]['model'] = MODEL_NAME.get(module, module)
                if module != 'src.model.run_grid':          # one command = one cell
                    begin_cell(gpu, _arg(cmd, '--split'), _arg(cmd, '--seed'))
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, bufsize=1)
                current[gpu] = proc
                for line in proc.stdout:
                    if module == 'src.model.run_grid':      # follow run_grid's own cells
                        m = HEADER.match(line) or SKIPPED.match(line)
                        if m:
                            begin_cell(gpu, m.group(1), m.group(2))
                    m = EPOCH.match(line)
                    if m:
                        where[gpu]['epoch'] = m.group(1)
                    print(f'{tag(gpu)} {line}', end='', flush=True)
                    log.write(line)
                    log.flush()
                    m = SAVED.search(line)
                    if m:
                        try:
                            auc = json.load(open(m.group(1)))['test_metrics'].get('auroc')
                            auc = f'{auc:.4f}'
                        except Exception:
                            auc = '?'
                        print(f'  ✓ {tag(gpu)} finished -- test AUROC {auc}   '
                              f'[{cells_done()}/{TOTAL_CELLS} cells complete]', flush=True)
                code_ = proc.wait()
                if code_ != 0 and not state['deadline']:
                    print(f'{tag(gpu)} !! exited {code_}: {" ".join(cmd[3:])}', flush=True)

    threads = [threading.Thread(target=worker, args=(g, q), daemon=True)
               for g, q in queues.items()]
    for t in threads:
        t.start()
    last_status = 0.0
    while any(t.is_alive() for t in threads):
        if time.time() - last_status >= STATUS_EVERY:
            last_status = time.time()
            parts = [f"GPU{g}: {w['model']} {w['split']} s{w['seed']} "
                     f"cell {w['cell']}/{w['total']} epoch {w['epoch']}"
                     for g, w in where.items()]
            print(f"\n=== STATUS {time.strftime('%H:%M')} | " + ' | '.join(parts)
                  + f' | {cells_done()}/{TOTAL_CELLS} complete | {hours_left():.1f} h left ===\n',
                  flush=True)
        if time.time() > DEADLINE and not state['deadline']:
            state['deadline'] = True
            print('\n*** 11-hour mark: stopping so this commit can save its output. '
                  'Unfinished cells are retrained next commit. ***\n', flush=True)
            for proc in list(current.values()):
                if proc.poll() is None:
                    proc.terminate()
        time.sleep(15)
    return state['deadline']


def deepdta_cmd(split, seed):
    return ['python', '-u', '-m', 'src.model.train_deepdta',
            '--split-dir', f'data/splits/{DATASET}/{split}', '--dataset', DATASET,
            '--split', split, '--task', 'binary', '--seed', str(seed),
            '--batch-size', str(DEEPDTA_BATCH), '--min-epochs', '10', '--epochs', '100',
            '--patience', '10',     # DeepDTA's own; DAVIS was trained with it
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS, '--skip-if-done']


def coldsite_cmd(splits):
    # patience is not a train.py flag; ColdSite-DTI's is fixed at 15 in run_training
    return ['python', '-u', '-m', 'src.model.run_grid', '--datasets', DATASET,
            '--splits', ','.join(splits), '--seeds', ','.join(map(str, SEEDS)),
            '--task', 'binary', '--epochs', '100', '--min-epochs', '10',
            '--batch-size', str(COLDSITE_BATCH), '--results-dir', RESULTS]


def hat_cmd(split, seed):
    return ['python', '-u', '-m', 'src.model.train_hyperattentiondti',
            '--split-dir', f'data/splits/{DATASET}/{split}', '--dataset', DATASET,
            '--split', split, '--seed', str(seed),
            '--batch-size', str(HAT_BATCH), '--accum-steps', str(HAT_ACCUM),
            '--patience', '15',
            '--min-epochs', '10', '--epochs', '100',
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS, '--skip-if-done']


def moltrans_cmd(split, seed):
    return ['python', '-u', '-m', 'src.model.train_moltrans',
            '--split-dir', f'data/splits/{DATASET}/{split}', '--dataset', DATASET,
            '--split', split, '--seed', str(seed),
            '--batch-size', str(MOLTRANS_BATCH), '--min-epochs', '10', '--epochs', '100',
            '--patience', '15',
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS, '--skip-if-done']


# Cheapest / fastest-signal model first, when it is part of this run.
MODEL_ORDER = ['deepdta', 'moltrans', 'coldsite_dti', 'hyperattentiondti']


def build_queue(splits):
    queue = []
    for model in MODEL_ORDER:
        if model not in MODELS:
            continue
        if model == 'deepdta':
            queue += [deepdta_cmd(split, seed) for split in splits for seed in SEEDS]
        elif model == 'coldsite_dti':
            queue.append(coldsite_cmd(splits))
        elif model == 'hyperattentiondti':
            queue += [hat_cmd(split, seed) for split in splits for seed in SEEDS]
        elif model == 'moltrans':
            queue += [moltrans_cmd(split, seed) for split in splits for seed in SEEDS]
    return queue


for gpu, splits in SHARE.items():
    print(f'GPU {gpu} -> {", ".join(splits)}  ({", ".join(MODELS)})')
print(f'{hours_left():.1f} h left before the self-stop')


## 8. Launch

In [ ]:
if hours_left() < 0.5:
    print(f'Skipped: only {hours_left():.1f} h left. Commit again with the restore cell set.')
else:
    queues = {gpu: build_queue(splits) for gpu, splits in SHARE.items()}
    cut_short = run_parallel(queues, 'grid')
    print('\n' + ('CUT SHORT by the deadline -- commit again with RESTORE_FROM set'
                  if cut_short else 'Queue finished on every GPU.'))


## 9. What landed

A cell counts as complete only when both its checkpoint and its results file exist. A
checkpoint alone is an interrupted cell, and the next commit retrains it.

In [ ]:
import statistics as st

complete = interrupted = 0
print(f"{'model':18s} {'split':12s} {'cells':>7s}   AUROC mean +- sd")
for model in MODELS:
    for split in SPLIT_SUBSET:
        values = []
        for seed in SEEDS:
            ckpt = checkpoint_path(RESULTS, DATASET, split, 'binary', seed, model=model)
            res = results_path(RESULTS, run_tag(DATASET, split, 'binary', seed), model=model)
            if os.path.exists(ckpt) and os.path.exists(res):
                complete += 1
                values.append(json.load(open(res))['test_metrics']['auroc'])
            elif os.path.exists(ckpt):
                interrupted += 1
        spread = (f'{st.mean(values):.4f} +- {st.stdev(values):.4f}' if len(values) > 1
                  else f'{values[0]:.4f}' if values else '--')
        print(f'{model:18s} {split:12s} {len(values):>3d}/{len(SEEDS):<3d}   {spread}')

RUN_COMPLETE = complete == TOTAL_CELLS
print(f'\ncomplete    : {complete} / {TOTAL_CELLS}')
print(f'interrupted : {interrupted}  (retrained next commit)')
print(f'time left   : {hours_left():.1f} h')
print("\nALL DONE for this run's settings." if RUN_COMPLETE else
      '\nNot done yet: download the output, update the restore dataset, commit again with RESTORE_FROM set.')


## 10. Take the results with you

Everything in `/kaggle/working` is saved as this version's output. The zip is the easy
download. Add its contents to your restore dataset (as a new version) before the next
commit, and keep a copy in Google Drive.

In [ ]:
ZIP_NAME = f'{DATASET}_{"_".join(MODELS)}_results.zip'
!cd {WORK} && rm -f {ZIP_NAME} && zip -qr {ZIP_NAME} results
print(f'{WORK}/{ZIP_NAME}  --  download from the Output panel')
